# 11.5 Labeled Memo Embedding + Topic Prototype

11번 노트북에서 저장한 `classification_detail` 샘플 라벨을 기반으로 embedding을 생성하고, 주제별 prototype embedding을 생성합니다.

이 노트북 결과가 있어야 12번 노트북에서 아직 주제분류되지 않은 memo를 ML/prototype 방식으로 분류할 수 있습니다.

In [ ]:
%sh
cd /Workspace/Users/jungryo.lee@lge.com/prj_TV_voc && git pull


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F
from pyspark.sql.window import Window

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import ml.memo_embedding as memo_embedding
import ml.topic_prototype as topic_prototype

importlib.reload(config_loader)
importlib.reload(memo_embedding)
importlib.reload(topic_prototype)

from common.config_loader import load_config, get_output_table
from ml.memo_embedding import build_and_save_memo_embeddings_ai_query
from ml.topic_prototype import (
    load_memo_embedding_df,
    summarize_embedding_table,
    build_topic_prototype_df,
    save_topic_prototypes,
)

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

MODEL_KEY = config["app"].get("model_key", "gpt_55")
MODEL_VERSION = config["llm"]["models"][MODEL_KEY]["model_version"]
PROMPT_VERSION = config["version"]["prompt_version"]
TAXONOMY_VERSION = config["version"]["taxonomy_version"]
EMBEDDING_MODEL = config["memo_embedding"].get("embedding_model", "databricks-bge-large-en")

print("settings =", config["path"]["settings"])
print("model_version =", MODEL_VERSION)
print("prompt_version =", PROMPT_VERSION)
print("taxonomy_version =", TAXONOMY_VERSION)
print("embedding_model =", EMBEDDING_MODEL)


In [ ]:
# 11번 결과 확인: classification_detail에 샘플 라벨이 있어야 합니다.
classification_detail_table = get_output_table(config, "classification_detail")

classified_df = (
    spark.table(classification_detail_table)
    .where(F.col("prompt_version") == PROMPT_VERSION)
    .where(F.col("taxonomy_version") == TAXONOMY_VERSION)
    .where(F.col("model_version") == MODEL_VERSION)
)

display(
    classified_df.groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "pred_topic_type")
    .agg(
        F.count("*").alias("row_cnt"),
        F.countDistinct("memo_id").alias("distinct_memo_id_cnt"),
        F.countDistinct("pred_topic").alias("distinct_topic_cnt"),
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", "pred_topic_type")
)


In [ ]:
# Labeled memo embedding 생성
# 처음 검증 시 LIMIT_ROWS를 작게 줄일 수 있습니다. 전체 실행은 None 권장.
LIMIT_ROWS = None
SKIP_EXISTING = True

embedding_result = build_and_save_memo_embeddings_ai_query(
    spark,
    config,
    input_table_key="classification_detail",
    output_table_key="memo_embedding",
    embedding_model=EMBEDDING_MODEL,
    limit_rows=LIMIT_ROWS,
    skip_existing=SKIP_EXISTING,
)

embedding_result


In [ ]:
# Embedding 결과 품질 확인
embedding_df = load_memo_embedding_df(
    spark,
    config,
    input_table_key="memo_embedding",
    embedding_model=EMBEDDING_MODEL,
    prompt_version=PROMPT_VERSION,
    taxonomy_version=TAXONOMY_VERSION,
    model_version=MODEL_VERSION,
)

# 같은 memo_id가 여러 번 쌓였을 경우 최신 1건만 prototype 생성에 사용합니다.
dedupe_window = Window.partitionBy(
    "cate_1_depth",
    "cate_2_depth",
    "sc_measurement",
    "memo_id",
    "prompt_version",
    "taxonomy_version",
    "model_version",
).orderBy(F.col("created_at").desc_nulls_last(), F.col("run_id").desc_nulls_last())

embedding_df = (
    embedding_df.withColumn("_rn", F.row_number().over(dedupe_window))
    .where(F.col("_rn") == 1)
    .drop("_rn")
)

summary = summarize_embedding_table(embedding_df)
display(summary["overall_df"])
display(summary["topic_df"])
display(summary["sparse_topic_df"])


In [ ]:
# Topic prototype 생성 및 저장
prototype_table = get_output_table(config, "topic_prototype")
MIN_TOPIC_MEMO_COUNT = config["topic_prototype"].get("min_topic_memo_count", 3)

prototype_df = build_topic_prototype_df(
    embedding_df,
    min_topic_memo_count=MIN_TOPIC_MEMO_COUNT,
)

prototype_count = prototype_df.count()
print("prototype_count =", prototype_count)

if spark.catalog.tableExists(prototype_table):
    spark.sql(f"""
    DELETE FROM {prototype_table}
    WHERE prompt_version = '{PROMPT_VERSION}'
      AND taxonomy_version = '{TAXONOMY_VERSION}'
      AND model_version = '{MODEL_VERSION}'
      AND embedding_model = '{EMBEDDING_MODEL}'
    """)
    save_mode = "append"
else:
    save_mode = "overwrite"

saved_table = save_topic_prototypes(
    prototype_df,
    config,
    output_table_key="topic_prototype",
    mode=save_mode,
)

print("saved_table =", saved_table)


In [ ]:
# 저장된 prototype 확인
display(
    spark.table(prototype_table)
    .where(F.col("prompt_version") == PROMPT_VERSION)
    .where(F.col("taxonomy_version") == TAXONOMY_VERSION)
    .where(F.col("model_version") == MODEL_VERSION)
    .where(F.col("embedding_model") == EMBEDDING_MODEL)
    .groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "pred_topic_type")
    .agg(
        F.count("*").alias("prototype_topic_cnt"),
        F.sum("prototype_distinct_memo_id_cnt").alias("prototype_label_memo_cnt"),
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", "pred_topic_type")
)
